# Branch-and-bound experiments (robust knapsack, n=80000 model)

Loads the **validation-set** (out-of-fold) and **test-set** NN predictions for the
n=80000 model, then runs **vanilla** B&B and **modified** B&B with a chosen
`tau_prune` / `tau_stop`. Edit the two risk levels in the *Margins* cell and re-run.

Modified-B&B cutoffs (MIN-sense; the solver minimizes `-value`, so `v_hat_min = -v_hat`):
- **prune** a node if `LB >= v_hat - q_prune`  (upper residual tail; risk `tau_prune` of
  pruning the optimal branch)
- **stop** early if `UB <= v_hat - q_stop`  (lower residual tail; risk `tau_stop` of a
  suboptimal early stop).
Set `RAW_CUTOFFS=True` to use `v_hat` directly (margins = 0).

In [1]:
import sys, pathlib, time, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
ROOT = pathlib.Path.cwd().parent; sys.path.insert(0, str(ROOT))
from nn.training import load_checkpoint, predict_denorm
from problems.robust_knapsack.branch_and_bound import solve_bnb
from problems.robust_knapsack.relaxation import build_relaxation_template
from problems.robust_knapsack.conformal import reconstruct_oof_predictions, compute_margin
DATA = ROOT/'data'/'robust_knapsack'; MODELS = ROOT/'models'/'robust_knapsack'
N_TRAIN, FOLDS = 80000, 4
pd.set_option('display.width', 200, 'display.max_columns', 30)

## 1. Load the n=80000 ensemble

In [2]:
models, scalers = [], []
for k in range(FOLDS):
    m, s, _ = load_checkpoint(MODELS/f'dnn_knapsack_n{N_TRAIN}_fold{k}.pt')
    models.append(m); scalers.append(s)
def ens_predict(X):
    return np.mean([predict_denorm(m, X, s) for m, s in zip(models, scalers)], axis=0)
print(f'loaded {FOLDS} fold checkpoints for n_train={N_TRAIN}')

loaded 4 fold checkpoints for n_train=80000


## 2. Validation-set (out-of-fold) predictions -> calibration residuals
Each training row is scored by the one fold model that did **not** train on it, so
`e = v_hat - Cost` is an honest held-out residual with no retraining.

In [3]:
X_val, cost_val, oof_pred = reconstruct_oof_predictions(
    DATA/f'train_{N_TRAIN}.csv', MODELS, N_TRAIN, folds=FOLDS)
e = oof_pred - cost_val   # over-prediction residuals on the validation set
print(f'validation set: {len(e)} rows   residual mean={e.mean():.4f}  std={e.std():.4f}'
      f'   [p1 {np.percentile(e,1):.3f}, p99 {np.percentile(e,99):.3f}]')

validation set: 80000 rows   residual mean=-0.0023  std=0.1794   [p1 -0.436, p99 0.409]


## 3. Test-set predictions
The ensemble `v_hat` for every test instance drives the modified B&B cutoffs.

In [4]:
test = pd.read_csv(DATA/'test_20000.csv')
test['Cost'] = pd.to_numeric(test['Cost'], errors='coerce')
test = test[np.isfinite(test['Cost'])].reset_index(drop=True)
mu_cols = [c for c in test.columns if c.startswith('Mu')]
s2_cols = [c for c in test.columns if c.startswith('Sigma2')]
feat = [c for c in test.columns if c != 'Cost']
test_pred = ens_predict(test[feat].values.astype(float))   # v_hat per test instance
print(f'{len(test)} test instances')

20000 test instances


## 4. Margins from your chosen risk levels
`compute_margin(oof_pred, cost, tau)` returns the `(1-tau)` quantile of `e`.
Prune uses the upper tail (`tau_prune`); stop uses the lower tail (`1 - tau_stop`).

In [24]:
tau_prune = 0.5      # <-- risk of pruning the optimal branch
tau_stop  = 0.5      # <-- risk of an early suboptimal stop
RAW_CUTOFFS = True   # True => use v_hat directly (both margins = 0)

if RAW_CUTOFFS:
    prune_margin = stop_margin = 0.0
else:
    prune_margin = compute_margin(oof_pred, cost_val, tau_prune)       # upper tail
    stop_margin  = compute_margin(oof_pred, cost_val, 1.0 - tau_stop)  # lower tail
print(f'prune_margin = {prune_margin:.4f}   stop_margin = {stop_margin:.4f}')

prune_margin = 0.0000   stop_margin = 0.0000


## 5. Run vanilla vs. modified B&B
`vanilla` = standard B&B (no cutoffs, exact). `modified` = NN cutoffs + early stop.
Increase `N_INSTANCES` for a tighter estimate (slower).

In [25]:
N_INSTANCES = 25
rng = np.random.default_rng(0)
sel = rng.choice(len(test), size=min(N_INSTANCES, len(test)), replace=False)
template = build_relaxation_template()

recs = []
for i in sel:
    mu = test.loc[i, mu_cols].values.astype(float)
    s2 = test.loc[i, s2_cols].values.astype(float)
    true_cost = float(test.loc[i, 'Cost']); vhat = float(test_pred[i])

    t0 = time.time()
    base = solve_bnb(mu, s2, prune_cutoff=np.inf, stop_cutoff=np.inf,
                     template=template, early_stop=False)
    base_t = time.time() - t0
    ref = max(true_cost, base.value)   # provable optimum

    t0 = time.time()
    nn = solve_bnb(mu, s2, prune_cutoff=prune_margin - vhat,
                   stop_cutoff=stop_margin - vhat, template=template, early_stop=True)
    nn_t = time.time() - t0
    unsafe = nn.value < ref - 1e-4
    recs.append(dict(instance=int(i), true_cost=ref, vhat=vhat,
        base_nodes=base.nodes_explored, base_time=base_t, base_value=base.value,
        nn_nodes=nn.nodes_explored, nn_time=nn_t, nn_value=nn.value, nn_status=nn.status,
        unsafe=bool(unsafe), opt_gap=(ref - nn.value) if unsafe else 0.0))
res = pd.DataFrame(recs)
res.head()

,instance,true_cost,vhat,base_nodes,base_time,base_value,nn_nodes,nn_time,nn_value,nn_status,unsafe,opt_gap
0,16253,26.368240,26.562531,531,0.340676,26.368240,0,0.000720,26.349239,solved,True,0.019001
1,18243,27.030048,26.890358,233,0.133067,27.030048,124,0.071370,26.996732,early_stop,True,0.033315
2,54,29.525259,29.568569,515,0.303732,29.525259,141,0.084455,29.402189,solved,True,0.123071
3,10211,29.194602,29.155764,335,0.198196,29.194602,220,0.126502,29.194602,early_stop,False,0.000000
4,10066,28.587686,28.574039,451,0.264233,28.587686,450,0.264030,28.587686,early_stop,False,0.000000


## 6. Speedup + safety summary

In [26]:
node_speedup = res.base_nodes.sum() / max(res.nn_nodes.sum(), 1)
time_speedup = res.base_time.sum() / max(res.nn_time.sum(), 1e-9)
print(f'node speedup = {node_speedup:.2f}x')
print(f'time speedup = {time_speedup:.2f}x')
print(f'unsafe rate  = {res.unsafe.mean()*100:.1f}%   (returned > optimum)')
print(f'mean gap | unsafe = {res.loc[res.unsafe, "opt_gap"].mean() if res.unsafe.any() else 0.0:.4f}')

node speedup = 2.51x
time speedup = 2.51x
unsafe rate  = 40.0%   (returned > optimum)
mean gap | unsafe = 0.1594


## 7. delta-optimal error vs. the validation residual tail
The early-stop mistake `gap >= delta` is bounded (per instance) by `v_hat - v >= delta`,
i.e. by the residual tail `Pr(e <= -delta)` on an *exchangeable* set. We compare the
observed mistake rate to the **validation** tail. NB: a train/test shift can make the
validation tail optimistic — calibrate on data exchangeable with the test stream.

In [8]:
gap = res.opt_gap.values
print(f'{"delta":>7} | observed Pr(gap>=d) | validation bound Pr(v_hat-v>=d)')
for dl in [0.1, 0.2, 0.3, 0.5]:
    obs = (gap >= dl).mean(); bound = (-e >= dl).mean()
    print(f'{dl:7.2f} | {obs*100:17.1f}% | {bound*100:.1f}%')

  delta | observed Pr(gap>=d) | validation bound Pr(v_hat-v>=d)
   0.10 |               0.0% | 28.8%
   0.20 |               0.0% | 13.4%
   0.30 |               0.0% | 5.0%
   0.50 |               0.0% | 0.4%
